# Advanced Ranking Algorithms

**Chapter 8: Feature Engineering for Soccer Predictions**

## Learning Objectives

- Understand why simple win percentage doesn't capture true team strength
- Implement the Colley Matrix using linear algebra
- Apply PageRank (from Google) to soccer rankings
- Build Elo rating system for dynamic team ratings
- Compare all three ranking methods
- Use rankings as features for prediction models

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
from datetime import datetime, timedelta

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

np.random.seed(42)

print("Advanced Ranking Toolkit Ready!")
print(f"NumPy version: {np.__version__}")
print(f"NetworkX version: {nx.__version__}")

## Why Advanced Rankings Matter

Simple metrics like goals scored and win percentage are useful, but they don't tell the whole story.

**Question**: Which team is stronger?
- Team A: 3 wins against bottom-5 opponents
- Team B: 3 wins against top-5 opponents

Both have the same record, but Team B is clearly stronger!

To capture this, we need **sophisticated ranking algorithms that account for strength of schedule**.

Think of it as creating a dynamic league table where you get more credit for beating stronger teams.

## Generate Match Data

In [ ]:
# Generate simulated tournament data
np.random.seed(42)

teams = ['USA', 'Netherlands', 'England', 'Spain', 'Germany', 'France', 
         'Brazil', 'Argentina', 'Japan', 'Australia', 'Sweden', 'Norway',
         'Canada', 'Italy', 'Portugal', 'Mexico']

matches = []
match_id = 1
start_date = datetime(2023, 7, 20)

# Generate round-robin style matches
for i, home_team in enumerate(teams):
    for j, away_team in enumerate(teams):
        if i < j:  # Each pair plays once
            match_date = start_date + timedelta(days=match_id * 3)
            
            # Simulate scores with some teams being stronger
            team_strength = {team: idx for idx, team in enumerate(teams)}
            strength_diff = team_strength[away_team] - team_strength[home_team]
            
            # Stronger teams (lower index) score more
            home_score = max(0, np.random.poisson(1.5 - strength_diff * 0.05))
            away_score = max(0, np.random.poisson(1.0 + strength_diff * 0.05))
            
            matches.append({
                'match_id': match_id,
                'match_date': match_date,
                'home_team': home_team,
                'away_team': away_team,
                'home_score': home_score,
                'away_score': away_score
            })
            
            match_id += 1

matches_df = pd.DataFrame(matches)
matches_df = matches_df.sort_values('match_date').reset_index(drop=True)

print(f"Generated {len(matches_df)} matches between {len(teams)} teams")
print(f"\nSample matches:")
matches_df.head(10)

---

# 1. The Colley Matrix: Linear Algebra Meets Soccer

The **Colley Matrix**, developed by Wesley Colley in 2002 for ranking college American football teams, is one of the most elegant ranking systems.

It's based on a beautiful piece of linear algebra: it sets up a system of equations where each team's rating depends on:
1. Its win-loss record
2. The ratings of the teams it has played against

It then solves these equations **simultaneously** to find ratings that are consistent with all the results.

### The Intuition

Your rating should be high if you win a lot, but it should be **even higher** if you win against strong opponents. Conversely, losing to a weak opponent should hurt your rating more than losing to a strong opponent.

### The Mathematics

The matrix equation: **Cr = b**

Where:
- **C** is the Colley matrix (representing the schedule—who played whom)
- **r** is the vector of ratings we're solving for
- **b** is the results vector (representing wins and losses)

In [ ]:
def calculate_colley_ratings(matches_df):
    """
    Calculate Colley Matrix ratings for all teams.
    
    Returns:
        DataFrame with teams and their Colley ratings
    """
    # Get unique teams and create index mapping
    teams = pd.unique(matches_df[["home_team", "away_team"]].values.ravel('K'))
    team_to_idx = {team: i for i, team in enumerate(teams)}
    num_teams = len(teams)
    
    # Initialize matrices
    C = np.zeros((num_teams, num_teams))
    b = np.zeros(num_teams)
    
    # Populate based on match results
    for _, row in matches_df.iterrows():
        i = team_to_idx[row['home_team']]
        j = team_to_idx[row['away_team']]
        
        # Diagonal: games played
        C[i, i] += 1
        C[j, j] += 1
        
        # Off-diagonal: opponents faced (negative)
        C[i, j] -= 1
        C[j, i] -= 1
        
        # Results vector: winner gets +1, loser gets -1
        if row['home_score'] > row['away_score']:
            b[i] += 1
            b[j] -= 1
        elif row['away_score'] > row['home_score']:
            b[j] += 1
            b[i] -= 1
        # Draws don't change b
    
    # Add 2 to diagonal (Colley method)
    # This ensures matrix is invertible and gives baseline rating of 0.5
    np.fill_diagonal(C, C.diagonal() + 2)
    
    # Solve the linear system
    ratings = np.linalg.solve(C, b)
    
    # Create results dataframe
    colley_df = pd.DataFrame({
        'team': teams,
        'colley_rating': ratings
    }).sort_values('colley_rating', ascending=False).reset_index(drop=True)
    
    return colley_df

# Calculate Colley ratings
colley_ratings = calculate_colley_ratings(matches_df)

print("Colley Ratings:")
print(colley_ratings)
print("\nInterpretation:")
print("  0.5 = Average team (50% expected win rate vs average opponents)")
print("  > 0.5 = Above average")
print("  < 0.5 = Below average")

### Understanding the Code

**1. Team-to-index mapping**: We need to translate team names to matrix row/column positions (0, 1, 2, ...)

**2. Initialize matrices**: C is square (num_teams × num_teams), b has length num_teams

**3. Populate matrices**: 
   - Diagonal of C counts games played (each team gets +1 per match)
   - Off-diagonal counts opponents faced (if teams i and j play, put -1 in C[i,j] and C[j,i])
   - Vector b tracks wins (+1) and losses (-1)

**4. Add 2 to diagonal**: Key part of Colley method - ensures matrix is invertible and gives baseline rating of 0.5

**5. Solve linear system**: NumPy finds the ratings that satisfy all constraints simultaneously!

In [ ]:
# Visualize Colley ratings
fig, ax = plt.subplots(figsize=(10, 8))

y_pos = np.arange(len(colley_ratings))
colors = ['darkgreen' if r > 0.5 else 'darkred' for r in colley_ratings['colley_rating']]

ax.barh(y_pos, colley_ratings['colley_rating'], color=colors, alpha=0.7)
ax.set_yticks(y_pos)
ax.set_yticklabels(colley_ratings['team'])
ax.set_xlabel('Colley Rating', fontsize=12)
ax.set_title('Team Rankings by Colley Matrix', fontsize=14, fontweight='bold')
ax.axvline(0.5, color='black', linestyle='--', linewidth=2, label='Average (0.5)')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

### Key Properties of Colley Ratings

✅ **Automatic strength of schedule adjustment**: Beating good teams increases your rating more

✅ **Interpretable**: 0.7 rating ≈ 70% expected win rate vs average teams

✅ **Unique solution**: The linear algebra guarantees one consistent answer

✅ **Fair**: All results are considered simultaneously, not sequentially

---

# 2. PageRank: From Google to the Goal

**PageRank** is the algorithm that made Google famous. Larry Page and Sergey Brin developed it in 1998 to rank web pages.

### The Core Idea

A web page is important if many other **important** pages link to it. A link from CNN.com is worth more than a link from a personal blog.

### Translation to Soccer

A win against a top-ranked team is a more significant "vote" for your quality than a win against a bottom-table team.

We model the tournament as a **network (graph)** where:
- Teams are **nodes**
- Matches are **directed edges**
- If Team A beats Team B, we draw an edge from B → A (B is "voting" for A's quality)

### The Formula

$$PR(A) = \frac{1-d}{N} + d \times \sum_{i} \frac{PR(T_i)}{C(T_i)}$$

Where:
- PR(A) is the PageRank of team A
- d is the damping factor (typically 0.85)
- N is the total number of teams
- T_i are the teams that A beat
- C(T_i) is the number of teams that T_i lost to

In [ ]:
def calculate_pagerank_ratings(matches_df, damping=0.85):
    """
    Calculate PageRank ratings for all teams.
    
    Args:
        matches_df: DataFrame with match results
        damping: Damping factor (default 0.85)
    
    Returns:
        DataFrame with teams and their PageRank scores
    """
    # Get unique teams
    teams = pd.unique(matches_df[["home_team", "away_team"]].values.ravel('K'))
    
    # Create directed graph
    G = nx.DiGraph()
    
    # Add all teams as nodes
    for team in teams:
        G.add_node(team)
    
    # Add edges based on results
    for _, row in matches_df.iterrows():
        home = row["home_team"]
        away = row["away_team"]
        home_score = row["home_score"]
        away_score = row["away_score"]
        
        if home_score > away_score:
            # Home wins: away "votes" for home
            # Weight by goal difference for more nuance
            G.add_edge(away, home, weight=home_score - away_score)
        elif away_score > home_score:
            # Away wins: home "votes" for away
            G.add_edge(home, away, weight=away_score - home_score)
        else:
            # Draw: both teams vote for each other with small weight
            G.add_edge(home, away, weight=0.5)
            G.add_edge(away, home, weight=0.5)
    
    # Calculate PageRank
    pagerank_scores = nx.pagerank(G, alpha=damping, weight='weight')
    
    # Create results dataframe
    pagerank_df = pd.DataFrame({
        'team': list(pagerank_scores.keys()),
        'pagerank_score': list(pagerank_scores.values())
    }).sort_values('pagerank_score', ascending=False).reset_index(drop=True)
    
    return pagerank_df

# Calculate PageRank ratings
pagerank_ratings = calculate_pagerank_ratings(matches_df)

print("PageRank Ratings:")
print(pagerank_ratings)
print("\nInterpretation:")
print("  Higher score = Stronger team (beat strong opponents)")
print("  Scores sum to 1.0 across all teams")

### Understanding the Code

**1. Create directed graph**: Teams are nodes, wins are directed edges

**2. Edge weights**: We weight edges by goal difference - not all wins are equal!

**3. Handle draws**: Both teams vote for each other with small weights

**4. Damping factor (α=0.85)**: 
   - 85% of rating comes from teams you beat
   - 15% is baseline given to all teams
   - Prevents teams with no wins from having zero rating

In [ ]:
# Visualize PageRank ratings
fig, ax = plt.subplots(figsize=(10, 8))

y_pos = np.arange(len(pagerank_ratings))
ax.barh(y_pos, pagerank_ratings['pagerank_score'], color='steelblue', alpha=0.7)
ax.set_yticks(y_pos)
ax.set_yticklabels(pagerank_ratings['team'])
ax.set_xlabel('PageRank Score', fontsize=12)
ax.set_title('Team Rankings by PageRank', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

### Key Properties of PageRank

✅ **Handles unbalanced schedules**: Teams that haven't played each other are still connected through the network

✅ **Transitive relationships**: If A beats B and B beats C, then A indirectly benefits from C's losses

✅ **Network effects**: The whole tournament structure matters, not just individual results

✅ **Proven algorithm**: Powers Google search - clearly works at scale!

---

# 3. Elo Ratings: The Chess Classic Adapted for Soccer

**Elo ratings**, originally developed by Arpad Elo for chess in the 1960s, are now used across many sports.

Unlike Colley and PageRank (which look at all results simultaneously), **Elo is dynamic**—it updates after each match based on expected vs actual outcome.

### The Core Idea

If a high-rated team beats a low-rated team, their rating barely changes (it was expected). But if a low-rated team upsets a high-rated team, there's a big swing in both ratings (it was unexpected).

This makes Elo **very responsive to recent performance**.

### The Formula

$$R_{new} = R_{old} + K \times (S - E)$$

Where:
- R_new is the new rating
- R_old is the old rating before the match
- K is a constant (typically 20-40) that controls how much ratings change
- S is the actual score (1 for win, 0.5 for draw, 0 for loss)
- E is the expected score based on the rating difference

### Expected Score Formula

$$E = \frac{1}{1 + 10^{(R_{opponent} - R_{team}) / 400}}$$

This creates an S-curve:
- 200-point advantage → 76% expected win rate
- 400-point advantage → 91% expected win rate

In [ ]:
def calculate_elo_ratings(matches_df, K=30, initial_rating=1500):
    """
    Calculate Elo ratings for all teams.
    
    Args:
        matches_df: DataFrame with match results (must be sorted by date)
        K: K-factor controlling rating volatility
        initial_rating: Starting rating for all teams
    
    Returns:
        rating_history: DataFrame with ratings after each match
        final_ratings: Series with final ratings for each team
    """
    ratings = {}
    rating_history = []
    
    # Process matches chronologically
    for _, match in matches_df.sort_values('match_date').iterrows():
        home = match['home_team']
        away = match['away_team']
        
        # Initialize ratings if first match
        if home not in ratings:
            ratings[home] = initial_rating
        if away not in ratings:
            ratings[away] = initial_rating
        
        # Calculate expected scores
        expected_home = 1 / (1 + 10 ** ((ratings[away] - ratings[home]) / 400))
        expected_away = 1 - expected_home
        
        # Determine actual scores
        if match['home_score'] > match['away_score']:
            actual_home, actual_away = 1, 0
        elif match['home_score'] < match['away_score']:
            actual_home, actual_away = 0, 1
        else:
            actual_home, actual_away = 0.5, 0.5
        
        # Update ratings
        new_home_rating = ratings[home] + K * (actual_home - expected_home)
        new_away_rating = ratings[away] + K * (actual_away - expected_away)
        
        # Store history
        rating_history.append({
            'match_date': match['match_date'],
            'match_id': match['match_id'],
            'home_team': home,
            'away_team': away,
            'home_rating_before': ratings[home],
            'away_rating_before': ratings[away],
            'home_rating_after': new_home_rating,
            'away_rating_after': new_away_rating,
            'home_score': match['home_score'],
            'away_score': match['away_score']
        })
        
        # Update ratings
        ratings[home] = new_home_rating
        ratings[away] = new_away_rating
    
    # Create final ratings dataframe
    final_ratings = pd.DataFrame({
        'team': list(ratings.keys()),
        'elo_rating': list(ratings.values())
    }).sort_values('elo_rating', ascending=False).reset_index(drop=True)
    
    return pd.DataFrame(rating_history), final_ratings

# Calculate Elo ratings
elo_history, elo_ratings = calculate_elo_ratings(matches_df, K=30)

print("Final Elo Ratings:")
print(elo_ratings)
print("\nInterpretation:")
print("  1500 = Average team (starting rating)")
print("  > 1500 = Above average")
print("  < 1500 = Below average")

### Understanding the Code

**1. Sequential processing**: MUST process matches chronologically - Elo is dynamic!

**2. Initial rating**: All teams start at 1500 (the "average")

**3. Expected score**: S-curve based on rating difference

**4. Rating update**: Proportional to surprise
   - Expected 0.8, won (1.0) → gain K × 0.2 points
   - Expected 0.8, lost (0.0) → lose K × 0.8 points (big loss!)

**5. K parameter**: Controls volatility
   - Higher K = more dramatic changes
   - Chess uses K=32 for established players
   - Soccer often uses K=20-40

In [ ]:
# Visualize final Elo ratings
fig, ax = plt.subplots(figsize=(10, 8))

y_pos = np.arange(len(elo_ratings))
colors = ['darkgreen' if r > 1500 else 'darkred' for r in elo_ratings['elo_rating']]

ax.barh(y_pos, elo_ratings['elo_rating'], color=colors, alpha=0.7)
ax.set_yticks(y_pos)
ax.set_yticklabels(elo_ratings['team'])
ax.set_xlabel('Elo Rating', fontsize=12)
ax.set_title('Team Rankings by Elo', fontsize=14, fontweight='bold')
ax.axvline(1500, color='black', linestyle='--', linewidth=2, label='Average (1500)')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

In [ ]:
# Visualize Elo rating evolution over time for top teams
top_teams = elo_ratings.head(5)['team'].tolist()

plt.figure(figsize=(14, 6))

for team in top_teams:
    team_history = elo_history[
        (elo_history['home_team'] == team) | (elo_history['away_team'] == team)
    ].copy()
    
    # Get rating after each match
    team_ratings = []
    match_dates = []
    
    for _, row in team_history.iterrows():
        if row['home_team'] == team:
            team_ratings.append(row['home_rating_after'])
        else:
            team_ratings.append(row['away_rating_after'])
        match_dates.append(row['match_date'])
    
    plt.plot(match_dates, team_ratings, marker='o', label=team, linewidth=2, markersize=4)

plt.xlabel('Date', fontsize=12)
plt.ylabel('Elo Rating', fontsize=12)
plt.title('Elo Rating Evolution (Top 5 Teams)', fontsize=14, fontweight='bold')
plt.axhline(1500, color='black', linestyle='--', alpha=0.5, label='Average')
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print("Notice how Elo ratings change dynamically after each match!")

### Key Properties of Elo

✅ **Dynamic**: Updates after each match, very responsive to recent form

✅ **Surprise-based**: Big upsets cause big rating swings

✅ **Predictive**: Can calculate win probability for any matchup

✅ **Widely used**: FIFA, FiveThirtyEight, and many others use Elo for soccer

---

# Comparing the Three Rankings

Let's compare all three methods side by side!

In [ ]:
# Merge all rankings
comparison = colley_ratings.merge(
    pagerank_ratings, on='team'
).merge(
    elo_ratings, on='team'
)

# Normalize scores to 0-1 range for comparison
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
comparison['colley_normalized'] = scaler.fit_transform(comparison[['colley_rating']])
comparison['pagerank_normalized'] = scaler.fit_transform(comparison[['pagerank_score']])
comparison['elo_normalized'] = scaler.fit_transform(comparison[['elo_rating']])

# Calculate average ranking
comparison['average_score'] = (comparison['colley_normalized'] + 
                                comparison['pagerank_normalized'] + 
                                comparison['elo_normalized']) / 3

comparison = comparison.sort_values('average_score', ascending=False)

print("Ranking Comparison (Normalized 0-1):")
print(comparison[['team', 'colley_normalized', 'pagerank_normalized', 
                  'elo_normalized', 'average_score']].round(3))

In [ ]:
# Visualize comparison
fig, ax = plt.subplots(figsize=(12, 10))

x = np.arange(len(comparison))
width = 0.25

ax.barh(x - width, comparison['colley_normalized'], width, label='Colley', alpha=0.8)
ax.barh(x, comparison['pagerank_normalized'], width, label='PageRank', alpha=0.8)
ax.barh(x + width, comparison['elo_normalized'], width, label='Elo', alpha=0.8)

ax.set_yticks(x)
ax.set_yticklabels(comparison['team'])
ax.set_xlabel('Normalized Score', fontsize=12)
ax.set_title('Ranking Method Comparison', fontsize=14, fontweight='bold')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

### Which Ranking is Best?

There's no single "best" method - each has strengths:

| Method | Strengths | Weaknesses | Best For |
|--------|-----------|------------|----------|
| **Colley** | Mathematically elegant, accounts for strength of schedule, unique solution | Static (doesn't update dynamically), requires complete season | End-of-season rankings, fair comparisons |
| **PageRank** | Network effects, handles transitive wins, proven at scale | Complex to explain, requires graph structure | Tournament rankings, identifying "hub" teams |
| **Elo** | Dynamic, predictive, responsive to recent form | Sensitive to K parameter, order-dependent | Live predictions, tracking form changes |

### Recommendation

**For prediction features**: Use **all three**! They capture different aspects of team strength:
- Colley: Overall quality adjusted for schedule
- PageRank: Network position and transitive strength
- Elo: Current form and momentum

Let your machine learning model decide which matters most!

## Summary

In this notebook, you learned:

✅ **Why simple win percentage fails** - doesn't account for opponent strength

✅ **Colley Matrix** - linear algebra approach, elegant and fair

✅ **PageRank** - network-based ranking from Google

✅ **Elo ratings** - dynamic, responsive, predictive

✅ **How to compare methods** - each has unique strengths

### Key Takeaways

**1. Strength of schedule matters**
- Beating strong teams should count more
- All three methods account for this

**2. Different perspectives**
- Colley: Mathematical fairness
- PageRank: Network effects
- Elo: Dynamic momentum

**3. Use as features**
- These rankings make excellent prediction features
- Combine multiple methods for best results

**4. Choose based on context**
- End-of-season: Colley or PageRank
- Live predictions: Elo
- Best approach: Use all three!

## Practice Exercises

1. **Modify K parameter**: Try different K values (10, 20, 30, 50) for Elo. How does it affect volatility?

2. **Home advantage in Elo**: Modify Elo to give home teams a 50-point boost in expected score calculation.

3. **Goal difference in Colley**: Weight wins by goal difference in the Colley matrix.

4. **Correlation analysis**: Calculate correlation between the three ranking methods.

5. **Prediction accuracy**: Use each ranking to predict match outcomes. Which is most accurate?

6. **Hybrid ranking**: Create a weighted average of all three methods.

7. **Temporal Elo**: Plot how Elo ratings change over time for all teams.

8. **PageRank variants**: Try different damping factors (0.7, 0.85, 0.95) for PageRank.